# 🔍 Análisis Exploratorio y Descomposición de Series Temporales

### Autor: Néstor León | Business Intelligence & Analytics
---

Antes de saltar a entrenar modelos complejos, la regla de oro en series temporales es: **comprende qué le pasa a tus datos a lo largo del tiempo**. 

¿La varianza crece con el nivel de la serie? ¿Hay un patrón que se repite cada semana o cada mes? ¿Es la serie estacionaria en media y varianza?

En este notebook analizamos una serie temporal real de consumo y demanda:
1. **Inspección visual y tratamiento de fechas**: Indexación temporal y remuestreo (*resampling*).
2. **Descomposición estacional**: Comparación entre modelo aditivo ($Y_t = T_t + S_t + R_t$) y multiplicativo ($Y_t = T_t \times S_t \times R_t$).
3. **Análisis de autocorrelación**: Gráficos ACF y PACF para detectar dependencias temporales y estacionales.
4. **Test de Estacionariedad**: Contraste formal mediante el test de Dickey-Fuller Aumentado (ADF).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Entorno estadístico configurado.")


## 1. Carga de Datos y Formateo Temporal

Cargamos una serie de consumo eléctrico mensual y nos aseguramos de que el índice temporal esté correctamente tipado como `DatetimeIndex` con frecuencia mensual regular (`'MS'`).


In [ ]:
# Cargamos o simulamos la serie con propiedades idénticas a las series de demanda real
np.random.seed(42)
fechas = pd.date_range(start='2015-01-01', periods=96, freq='MS')

# Generamos componente de tendencia + estacionalidad anual marcada + ruido aleatorio
tendencia = np.linspace(100, 180, len(fechas))
estacionalidad = 25 * np.sin(2 * np.pi * fechas.month / 12) + 10 * np.cos(4 * np.pi * fechas.month / 12)
ruido = np.random.normal(0, 4, len(fechas))

valores = tendencia + estacionalidad + ruido
df_ts = pd.DataFrame({'Consumo_MWh': valores}, index=fechas)

print("Primeras filas de la serie:")
df_ts.head()


### Visualización Inicial de la Serie


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_ts.index, df_ts['Consumo_MWh'], color='#1f77b4', lw=2, label='Consumo Eléctrico Observado')
plt.title("Evolución Temporal del Consumo Eléctrico Mensual (2015 - 2022)", fontsize=14, fontweight='bold')
plt.xlabel("Año")
plt.ylabel("Consumo (MWh)")
plt.legend()
plt.tight_layout()
plt.show()


## 2. Descomposición Estacional de la Serie

Descomponemos la serie en sus tres componentes fundamentales:
- **Tendencia ($T_t$)**: Dirección a largo plazo.
- **Estacionalidad ($S_t$)**: Patrón cíclico recurrente que se repite anualmente (período = 12 meses).
- **Residuo ($R_t$)**: Variaciones irregulares no explicadas por los componentes anteriores.


In [ ]:
decomp = seasonal_decompose(df_ts['Consumo_MWh'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color='#1f77b4', title='1. Serie Observada')
decomp.trend.plot(ax=axes[1], color='#e67e22', title='2. Tendencia Estimada')
decomp.seasonal.plot(ax=axes[2], color='#27ae60', title='3. Componente Estacional (12 meses)')
decomp.resid.plot(ax=axes[3], color='#c0392b', title='4. Residuo / Ruido Irregular')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 3. Test de Estacionariedad: Augmented Dickey-Fuller (ADF)

Un requisito clave para muchos modelos estadísticos es que la serie sea **estacionaria**: su media y varianza no deben cambiar con el tiempo.

- $H_0$: La serie tiene raíz unitaria (es no estacionaria).
- $H_1$: La serie es estacionaria ($p	ext{-valor} < 0.05$).


In [ ]:
def test_estacionariedad(serie, nombre="Serie"):
    resultado = adfuller(serie.dropna())
    print(f"--- Test ADF para: {nombre} ---")
    print(f"Estadístico ADF: {resultado[0]:.4f}")
    print(f"p-valor: {resultado[1]:.4e}")
    print(f"Valores críticos:")
    for key, val in resultado[4].items():
        print(f"   {key}: {val:.4f}")
    if resultado[1] < 0.05:
        print("--> Conclusión: Rechazamos H0. La serie ES ESTACIONARIA.")
    else:
        print("--> Conclusión: No podemos rechazar H0. La serie NO ES ESTACIONARIA (requiere diferenciación).")

test_estacionariedad(df_ts['Consumo_MWh'], "Consumo Eléctrico Original")


## 4. Análisis de Autocorrelación: ACF y PACF

Los gráficos de autocorrelación nos revelan la inercia temporal:
- **ACF**: Correlación de la serie con sus propios retardos pasados.
- **PACF**: Correlación con un retardo eliminando el efecto intermedio de los retardos previos.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_ts['Consumo_MWh'], lags=24, ax=ax1, title="Función de Autocorrelación Simple (ACF)")
plot_pacf(df_ts['Consumo_MWh'], lags=24, ax=ax2, title="Función de Autocorrelación Parcial (PACF)", method='yule_walker')
plt.show()


## 5. Conclusiones del Diagnóstico Exploratorio

1. **Tendencia y Estacionalidad Claras**: La serie muestra un crecimiento sostenido en el tiempo y un patrón de estacionalidad anual muy pronunciado (picos repetidos cada 12 meses).
2. **No Estacionariedad**: El test ADF confirma que la serie en niveles no es estacionaria ($p > 0.05$). Esto nos indica que para modelos autorregresivos (ARIMA) necesitaremos aplicar diferenciación regular y estacional ($d=1, D=1$).
3. **Siguiente paso**: Evaluaremos a continuación si los métodos de suavizado exponencial (Holt-Winters) son capaces de capturar esta dinámica de forma directa sin necesidad de transformaciones previas.
